In [9]:
from sklearn.cluster import AgglomerativeClustering
from sklearn.neighbors import kneighbors_graph
import numpy as np
from scipy.cluster.hierarchy import linkage
from sklearn.neighbors import kneighbors_graph
from sklearn.cluster import AgglomerativeClustering
import numpy as np
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage
import numpy as np
from src.clustering import get_average_random_pair_similarity, check_coarse_elbow, plot_elbow_curve, plot_hierarchy_tree, build_dynamic_hierarchy, get_cluster_metrics_table, save_artifacts, apply_hierarchy_to_queries, format_hierarchical_metrics

In [10]:
# 1. Load the embeddings
doc_emb = np.load("../data/artifacts/doc_emb.npy")
query_emb = np.load("../data/artifacts/query_emb.npy")

# 2. Calculate baseline similarity
avg_sim_bge = get_average_random_pair_similarity(doc_emb, num_pairs=10000)
print(f"BGE EMBEDDING: Average cosine similarity for 10,000 random pairs: {avg_sim_bge:.4f}")

BGE EMBEDDING: Average cosine similarity for 10,000 random pairs: 0.3736


In [11]:
# Check elbow on a sample to decide our Coarse K
k_options = [10, 20, 30, 40, 50, 60, 70, 80]
elbow_df = check_coarse_elbow(doc_emb, k_options, sample_size=50000)

# Plot it
fig = plot_elbow_curve(elbow_df)
fig.show()

Subsampling 50,000 docs out of 101,962 for Elbow check...
Calculating inertias...
  k=10 -> Inertia: 28495.9258
  k=20 -> Inertia: 27597.7305
  k=30 -> Inertia: 27101.3281
  k=40 -> Inertia: 26743.4902
  k=50 -> Inertia: 26429.1484
  k=60 -> Inertia: 26192.2441
  k=70 -> Inertia: 25965.0957
  k=80 -> Inertia: 25764.3281


In [12]:
CHOSEN_COARSE_K = 20  
fine_options = [2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 15, 18, 20, 22, 25]

# Run the full hierarchy
results = build_dynamic_hierarchy(
    doc_emb=doc_emb, 
    chosen_coarse_k=CHOSEN_COARSE_K, 
    fine_k_options=fine_options
)

# View the new Dendrogram-style Icicle Tree
tree_fig = plot_hierarchy_tree(results["metrics_df"])
tree_fig.show()

L2 Normalizing complete document dataset...
Fitting full dataset to Coarse K=20...

Optimizing Fine K for each of the 20 coarse clusters...

Full Hierarchy Build Complete!


In [13]:
# Predict query mappings
q_coarse, q_hierarchical = apply_hierarchy_to_queries(
    query_emb=query_emb,
    coarse_model=results["coarse_model"],
    fine_models=results["fine_models"]
)

# Save everything to disk
save_artifacts(
    save_dir="../data/artifacts/clustering_output",
    doc_labels=results["doc_hierarchical_labels"],
    query_labels=q_hierarchical,
    coarse_model=results["coarse_model"],
    fine_models=results["fine_models"]
)

All artifacts successfully saved to ../data/artifacts/clustering_output/


In [14]:
from sklearn.preprocessing import normalize

# 1. L2-Normalize your notebook's embeddings (Metrics table expects cosine space!)
doc_emb_normalized = normalize(doc_emb, norm='l2')

# 2. Extract the coarse labels from the results dictionary you generated earlier
my_coarse_labels = results["doc_coarse_labels"]

# Extract the raw metrics table from your pipeline results
raw_df = results["metrics_df"]

# Apply the formatting and display it
formatted_table = format_hierarchical_metrics(raw_df)

# The display() function will render the styled HTML table in Jupyter
display(formatted_table)

In [15]:
import plotly.express as px

# 1. Extract df_fine
df_fine = results["metrics_df"][results["metrics_df"]["Level"] == "Fine"].copy()

# ==========================================
# Chart 1: Fine Clusters per Coarse Cluster
# ==========================================
# Extract the Coarse Parent (e.g., "C0" from "C0_F1")
df_fine["Coarse Parent"] = df_fine["Cluster ID"].apply(lambda x: x.split("_")[0])

# Count how many fine clusters belong to each coarse parent
fine_cluster_counts = df_fine.groupby("Coarse Parent").size().reset_index(name="Fine_Count")

# CRITICAL FIX: Cast to integer and get explicitly sorted unique values
fine_cluster_counts["Fine_Count"] = fine_cluster_counts["Fine_Count"].astype(int)
sorted_x_values = sorted(fine_cluster_counts["Fine_Count"].unique())

# Plot the distribution
fig1 = px.histogram(
    fine_cluster_counts, 
    x="Fine_Count", 
    title="Distribution: How many Fine Clusters per Coarse Cluster?",
    labels={"Fine_Count": "Number of Fine Clusters"},
    text_auto=True,
    color_discrete_sequence=['#D6BEFA']
)

# Update layout to explicitly order by our sorted integer array
fig1.update_layout(
    yaxis_title="Number of Coarse Clusters", 
    xaxis=dict(
        type='category',                     
        categoryorder='array',              # Change from 'category ascending' to 'array'
        categoryarray=sorted_x_values       # Pass the properly sorted [5, 8, 10, 12] array
    )
)
fig1.show()

In [16]:
# ==========================================
# Chart 2: Documents per Fine Cluster
# ==========================================
# Plot the distribution of cluster sizes (How many documents end up in a fine cluster?)
fig2 = px.histogram(
    df_fine, 
    x="Size", 
    title="Distribution: Number of Documents per Fine Cluster",
    labels={"Size": "Documents per Fine Cluster"},
    nbins=30, # Adjust this if you want wider/narrower bins
    color_discrete_sequence=['#CB94F7']
)
fig2.update_layout(yaxis_title="Number of Fine Clusters")
fig2.show()

In [17]:
import pandas as pd

query_df = pd.read_parquet("../data/interim/questions.parquet")
docs_df = pd.read_parquet("../data/interim/nodes.parquet")

In [18]:
import pandas as pd
import numpy as np
import ast
import plotly.express as px
import os
import joblib

# Define the directory where you saved your execution results
save_dir = "../data/artifacts/clustering_output"

# 1. Load the binary array assignments
doc_hierarchical_labels = np.load(os.path.join(save_dir, "doc_hierarchical_labels.npy"), allow_pickle=True)
query_hierarchical_labels = np.load(os.path.join(save_dir, "query_hierarchical_labels.npy"), allow_pickle=True)

# 2. Load the compressed clustering model architectures
coarse_model = joblib.load(os.path.join(save_dir, "kmeans_coarse.pkl"))
fine_models_dict = joblib.load(os.path.join(save_dir, "kmeans_fine_dict.pkl"))

print("All pipeline files successfully loaded into active session memory.")

# 1. Map Node IDs to their Cluster Labels
# We assume results["doc_coarse_labels"] matches the index/order of docs_df
node_to_coarse = dict(zip(docs_df['node_id'], results['doc_coarse_labels']))
node_to_fine = dict(zip(docs_df['n' \
'ode_id'], results['doc_hierarchical_labels']))

# Helper function to safely parse lists and count unique clusters
def count_spanned_clusters(row):
    gold_nodes = row['gold_node_ids']
    
    # Safely evaluate if pandas loaded the list as a string
    if isinstance(gold_nodes, str):
        try:
            gold_nodes = ast.literal_eval(gold_nodes)
        except:
            gold_nodes = [] # Fallback
            
    # Collect all unique clusters these gold nodes belong to
    coarse_set = set()
    fine_set = set()
    
    for node in gold_nodes:
        if node in node_to_coarse:
            coarse_set.add(node_to_coarse[node])
            fine_set.add(node_to_fine[node])
            
    return pd.Series({
        'spanned_coarse_count': len(coarse_set),
        'spanned_fine_count': len(fine_set)
    })

# 2. Apply the counting logic to your query dataframe
print("Calculating cluster spread for queries...")
query_df[['spanned_coarse_count', 'spanned_fine_count']] = query_df.apply(count_spanned_clusters, axis=1)

# Filter out any rows that didn't match or where n_hops isn't what we want
df_plot = query_df[query_df['n_hops'].isin([2, 3, 4])].copy()

All pipeline files successfully loaded into active session memory.
Calculating cluster spread for queries...


In [19]:
# ==========================================
# Chart 1: Coarse Cluster Spread per Query
# ==========================================
fig1 = px.histogram(
    df_plot,
    x='spanned_coarse_count',
    facet_col='n_hops',          # Creates separate side-by-side plots for 2, 3, 4
    color='n_hops',              # Colors them differently for easy reading
    title="Coarse Fragmentation: How many distinct Coarse Clusters do Gold Nodes span?",
    labels={'spanned_coarse_count': 'Distinct Coarse Clusters', 'n_hops': 'Hops'},
    text_auto=True
)

# Force X-axis to be discrete categorical buckets (1, 2, 3...)
fig1.update_xaxes(type='category', categoryorder='category ascending')
fig1.update_layout(showlegend=False, yaxis_title="Number of Queries")
fig1.show()


# ==========================================
# Chart 2: Fine Cluster Spread (ONLY for Queries contained in 1 Coarse Cluster)
# ==========================================
# Filter strictly for queries where all gold nodes fell into a SINGLE coarse cluster
df_contained = df_plot[df_plot['spanned_coarse_count'] == 1].copy()

fig2 = px.histogram(
    df_contained,
    x='spanned_fine_count',
    facet_col='n_hops',
    color='n_hops',
    title="Fine Fragmentation: Spanned Fine Clusters (For Gold Nodes contained in 1 Coarse Cluster)",
    labels={'spanned_fine_count': 'Distinct Fine Clusters', 'n_hops': 'Hops'},
    text_auto=True
)

fig2.update_xaxes(type='category', categoryorder='category ascending')
fig2.update_layout(showlegend=False, yaxis_title="Number of Queries")
fig2.show()

In [20]:
import ast
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.preprocessing import normalize

# ==========================================
# 1. Extract Centroids & Compute All Ranks
# ==========================================
print("Extracting centroids and calculating query-to-cluster ranks...")

# Coarse Centroids
coarse_centroids = results["coarse_model"].cluster_centers_

# Fine Centroids (Flattened globally for absolute ranking)
fine_ids = []
fine_centroids_list = []
for c_id, f_model in results["fine_models"].items():
    for f_label, centroid in enumerate(f_model.cluster_centers_):
        fine_ids.append(f"C{c_id}_F{f_label}")
        fine_centroids_list.append(centroid)
        
fine_centroids = np.array(fine_centroids_list)

# Normalize queries for Cosine Similarity
queries_norm = normalize(query_emb, norm='l2')

# Matrix multiplication for instant Cosine Similarities
coarse_sims = queries_norm @ coarse_centroids.T
fine_sims = queries_norm @ fine_centroids.T

# Magic numpy trick to convert similarities to Ranks (1 = highest similarity)
coarse_ranks_matrix = np.argsort(np.argsort(-coarse_sims, axis=1), axis=1) + 1
fine_ranks_matrix = np.argsort(np.argsort(-fine_sims, axis=1), axis=1) + 1


# ==========================================
# 2. Map Gold Nodes to their Ranks
# ==========================================
print("Mapping gold nodes to their cluster ranks...")

# Dictionaries to quickly look up which cluster a node belongs to
node_to_coarse = dict(zip(docs_df['node_id'], results['doc_coarse_labels']))
node_to_fine = dict(zip(docs_df['node_id'], results['doc_hierarchical_labels']))

rank_records = []

for idx, row in query_df.iterrows():
    n_hops = row['n_hops']
    if n_hops not in [2, 3, 4]: 
        continue
        
    gold_nodes = row['gold_node_ids']
    if isinstance(gold_nodes, str):
        try:
            gold_nodes = ast.literal_eval(gold_nodes)
        except:
            gold_nodes = []
            
    # Find unique clusters that contain the gold nodes for this specific query
    coarse_set = set()
    fine_set = set()
    for node in gold_nodes:
        if node in node_to_coarse:
            coarse_set.add(node_to_coarse[node])
            fine_set.add(node_to_fine[node])
            
    # Record the rank of EVERY unique coarse cluster required to answer the query
    for c_id in coarse_set:
        c_rank = coarse_ranks_matrix[idx, c_id]
        rank_records.append({'n_hops': n_hops, 'Level': 'Coarse', 'Rank': c_rank})
        
    # Record the rank of EVERY unique fine cluster required
    for f_id in fine_set:
        f_matrix_idx = fine_ids.index(f_id)
        f_rank = fine_ranks_matrix[idx, f_matrix_idx]
        rank_records.append({'n_hops': n_hops, 'Level': 'Fine', 'Rank': f_rank})

df_ranks = pd.DataFrame(rank_records)


# ==========================================
# 3. Plot Histograms
# ==========================================
print("Generating Plots...")

# Chart 1: Coarse Cluster Ranks
df_coarse_ranks = df_ranks[df_ranks['Level'] == 'Coarse']
fig1 = px.histogram(
    df_coarse_ranks,
    x='Rank',
    facet_col='n_hops',
    color='n_hops',
    title="Coarse Cluster Retrieval: Rank of Gold Clusters (1 = Best)",
    labels={'Rank': 'Rank of Gold Coarse Cluster', 'n_hops': 'Hops'},
    nbins=40 # Assuming you have roughly 40 coarse clusters
)
fig1.update_layout(showlegend=False, yaxis_title="Count of Gold Clusters")
fig1.show()


# Chart 2: Fine Cluster Ranks
df_fine_ranks = df_ranks[df_ranks['Level'] == 'Fine']
fig2 = px.histogram(
    df_fine_ranks,
    x='Rank',
    facet_col='n_hops',
    color='n_hops',
    title="Fine Cluster Retrieval: Global Rank of Gold Fine Clusters (Out of all ~400)",
    labels={'Rank': 'Global Rank of Gold Fine Cluster', 'n_hops': 'Hops'},
    nbins=50
)
fig2.update_layout(showlegend=False, yaxis_title="Count of Gold Clusters")
fig2.show()

Extracting centroids and calculating query-to-cluster ranks...
Mapping gold nodes to their cluster ranks...
Generating Plots...


In [87]:
import ast
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.preprocessing import normalize

# ---- Config ----
CLUSTER_BUDGET = 25      # docs the routing agent retrieves (top cosine within oracle clusters)
FULL_BUDGET    = 100     # docs standard vector RAG retrieves from the whole corpus
HOPS           = [2, 3, 4]
SPLIT_COL, VAL_SPLIT = "split", "validation"

# ---- Embeddings in cosine space (your arrays are NOT pre-normalized) ----
doc_norm   = normalize(doc_emb,   norm="l2").astype(np.float32)
query_norm = normalize(query_emb, norm="l2").astype(np.float32)
N_DOCS = len(doc_norm)

# ---- Fine label per doc index, encoded as int codes for fast set membership ----
doc_fine_labels = np.asarray(results["doc_hierarchical_labels"], dtype=object)
_, doc_label_codes = np.unique(doc_fine_labels, return_inverse=True)   # code per doc index

# Precompute the doc indices belonging to each fine cluster (so routing is a lookup, not a scan)
cluster_to_doc_idx = {}
for i, code in enumerate(doc_label_codes):
    cluster_to_doc_idx.setdefault(int(code), []).append(i)
cluster_to_doc_idx = {c: np.asarray(v) for c, v in cluster_to_doc_idx.items()}

# ---- node_id (string) -> row index in doc_emb (docs_df order == doc_emb order) ----
node_id_to_idx = {nid: i for i, nid in enumerate(docs_df["node_id"].tolist())}

def parse_gold(g):
    if isinstance(g, str):
        try: g = ast.literal_eval(g)
        except Exception: g = []
    return list(g) if g is not None else []

print(f"Corpus: {N_DOCS:,} docs across {len(cluster_to_doc_idx)} fine clusters.")

Corpus: 101,962 docs across 228 fine clusters.


In [88]:
# ---- Select validation queries (keep original index = query_emb row) ----
qdf = query_df
if SPLIT_COL in qdf.columns:
    qdf = qdf[qdf[SPLIT_COL] == VAL_SPLIT]
    print(f"Validation queries: {len(qdf)}")
else:
    print(f"No '{SPLIT_COL}' column — using ALL {len(qdf)} queries.")
qdf = qdf[qdf["n_hops"].isin(HOPS)]

# ---- Phase A: collect valid queries (gold mapped to doc indices) ----
items, dropped = [], 0
for qi, row in qdf.iterrows():
    gold = parse_gold(row["gold_node_ids"])
    gidx = [node_id_to_idx[n] for n in gold if n in node_id_to_idx]
    dropped += len(gold) - len(gidx)
    if gidx:
        items.append((qi, row["n_hops"], np.asarray(gidx)))
val_qis = np.asarray([it[0] for it in items])
print(f"{len(items)} queries with usable gold; {dropped} gold nodes not in corpus (excluded).")

# ---- Phase B: standard vector RAG top-FULL_BUDGET (chunked matmul over full corpus) ----
full_topsets, CH = {}, 64
for s in range(0, len(val_qis), CH):
    chunk = val_qis[s:s+CH]
    sims = doc_norm @ query_norm[chunk].T
    for j, qi in enumerate(chunk):
        col = sims[:, j]
        part = np.argpartition(-col, FULL_BUDGET)[:FULL_BUDGET]
        full_topsets[qi] = set(part.tolist())

# ---- Phase C: oracle routing — top-CLUSTER_BUDGET PER gold cluster, then union ----
records = []
for qi, n_hops, gidx in items:
    gold_set, n_gold = set(gidx.tolist()), len(gidx)
    q = query_norm[qi]

    full_hits = len(full_topsets[qi] & gold_set)

    gold_codes = set(doc_label_codes[gidx].tolist())     # the fine clusters the router must guess
    retrieved = set()
    for c in gold_codes:
        c_doc_idx = cluster_to_doc_idx[c]
        c_sims = doc_norm[c_doc_idx] @ q
        k = min(CLUSTER_BUDGET, len(c_doc_idx))
        top_c = (c_doc_idx[np.argpartition(-c_sims, k-1)[:k]]
                 if k < len(c_doc_idx) else c_doc_idx)
        retrieved.update(top_c.tolist())

    oracle_hits = len(retrieved & gold_set)
    records.append({
        "n_hops": n_hops,
        "n_gold": n_gold,
        "n_gold_clusters": len(gold_codes),
        "docs_retrieved": len(retrieved),                # ≈ CLUSTER_BUDGET × #clusters
        "oracle_cluster_recall": oracle_hits / n_gold,
        "full_rag_recall":       full_hits  / n_gold,
    })

res_df = pd.DataFrame(records)
print(f"\nAvg docs pulled by oracle routing: {res_df['docs_retrieved'].mean():.1f} "
      f"(vs {FULL_BUDGET} for vector RAG) across "
      f"{res_df['n_gold_clusters'].mean():.1f} clusters/query on average.")

Validation queries: 2417
2417 queries with usable gold; 0 gold nodes not in corpus (excluded).

Avg docs pulled by oracle routing: 53.7 (vs 100 for vector RAG) across 2.1 clusters/query on average.


In [89]:
cols = ["oracle_cluster_recall", "full_rag_recall"]
agg = res_df.groupby("n_hops")[cols].mean().mul(100)
agg = agg.join(res_df.groupby("n_hops")[["n_gold", "n_gold_clusters", "docs_retrieved"]].mean())
agg.loc["All", cols] = res_df[cols].mean().mul(100).values
agg.loc["All", ["n_gold", "n_gold_clusters", "docs_retrieved"]] = \
    res_df[["n_gold", "n_gold_clusters", "docs_retrieved"]].mean().values
print(agg.round(2))

plot_df = (agg.loc[:, cols].reset_index()
           .melt(id_vars="n_hops", var_name="Method", value_name="Recall (%)"))
plot_df["Method"] = plot_df["Method"].map({
    "oracle_cluster_recall": f"Oracle Cluster Routing ({CLUSTER_BUDGET}/cluster)",
    "full_rag_recall":       f"Standard Vector RAG (top-{FULL_BUDGET})",
})
plot_df["n_hops"] = plot_df["n_hops"].astype(str)

fig = px.bar(
    plot_df, x="n_hops", y="Recall (%)", color="Method",
    barmode="group", text_auto=".1f",
    color_discrete_sequence=["#27ae60", "#7f8c8d"],
    title="Recall Ceiling: Oracle Per-Cluster Routing vs. Standard Vector RAG",
    labels={"n_hops": "Query Hops"},
)
fig.update_layout(yaxis_range=[0, 100], legend_title_text="")
fig.show()

        oracle_cluster_recall  full_rag_recall  n_gold  n_gold_clusters  \
n_hops                                                                    
2                       81.35            77.52    2.00             1.71   
3                       78.03            73.33    3.00             2.34   
4                       63.77            48.58    4.00             3.16   
All                     77.36            71.35    2.65             2.15   

        docs_retrieved  
n_hops                  
2                42.65  
3                58.55  
4                78.95  
All              53.73  


In [57]:
import numpy as np
import pandas as pd
import plotly.express as px

# ── STEP 1: PRECOMPUTE CLUSTER CENTROIDS ─────────────────────────────────────
print("Computing representation vectors (centroids) for all fine clusters...")
num_clusters = len(cluster_to_doc_idx)
embedding_dim = doc_norm.shape[1]

cluster_centroids = np.zeros((num_clusters, embedding_dim), dtype=np.float32)
for c, idxs in cluster_to_doc_idx.items():
    # Mean of all document embeddings assigned to this fine cluster
    centroid = doc_norm[idxs].mean(axis=0)
    # Re-normalize to L2 space so dot product directly yields cosine similarity
    cluster_centroids[c] = centroid / np.linalg.norm(centroid)


# ── STEP 2: CALCULATE RANK OF FARTHEST GOLD CLUSTER PER QUERY ──────────────────
rank_records = []

for qi, n_hops, gidx in items:
    q = query_norm[qi]
    
    # Get the unique fine cluster codes that contain gold documents for this query
    gold_codes = np.unique(doc_label_codes[gidx])
    
    # Compute cosine similarity between the query and ALL cluster centroids
    sims_to_all_clusters = cluster_centroids @ q
    
    # Sort cluster indices by similarity in descending order (highest similarity first)
    sorted_cluster_ranks = np.argsort(-sims_to_all_clusters)
    
    # Locate the position (rank) of our required gold clusters within the sorted array
    # Rank 0 means it was the absolute closest cluster to the query vector
    gold_cluster_ranks = np.where(np.isin(sorted_cluster_ranks, gold_codes))[0]
    
    # Find the maximum (farthest) rank required to capture all gold clusters for this query
    farthest_rank = int(np.max(gold_cluster_ranks))
    
    rank_records.append({
        "query_index": qi,
        "n_hops": n_hops,
        "farthest_cluster_rank": farthest_rank,
        "num_gold_clusters": len(gold_codes)
    })

rank_df = pd.DataFrame(rank_records)


# ── STEP 3: PRINT SUMMARY STATS PER HOP TYPE ─────────────────────────────────
print("\n--- Farthest Cluster Rank Summary Statistics by Hop Complexity ---")
summary_stats = rank_df.groupby("n_hops")["farthest_cluster_rank"].agg(
    ["count", "mean", "median", "max", lambda x: np.percentile(x, 90), lambda x: np.percentile(x, 95), lambda x: np.percentile(x, 99)]
).rename(columns={"<lambda_0>": "90th_percentile"})
print(summary_stats.round(1))


# ── STEP 4: PLOT DISTRIBUTION HISTOGRAM VIA PLOTLY ───────────────────────────
# Convert n_hops to a string category to ensure clean categorical split in the plot
rank_df["n_hops_str"] = rank_df["n_hops"].astype(str) + " Hops"

fig = px.histogram(
    rank_df,
    x="farthest_cluster_rank",
    color="n_hops_str",
    facet_row="n_hops_str",
    marginal="box",  # Adds a matching boxplot above each histogram for clear quartile trends
    nbins=50,
    title="Distribution of Farthest Fine Cluster Rank Needed to Resolve Queries",
    labels={
        "farthest_cluster_rank": "Farthest Required Cluster Rank (0 = Closest Cluster)",
        "n_hops_str": "Query Complexity"
    },
    color_discrete_sequence=px.colors.qualitative.Safe,
)

fig.update_layout(
    height=700,
    showlegend=False,
    hovermode="x unified",
    bargap=0.05
)

# Enforce starting the x-axis explicitly from rank 0
fig.update_xaxes(matches='x', range=[-1, rank_df["farthest_cluster_rank"].max() + 5])
fig.show()

Computing representation vectors (centroids) for all fine clusters...

--- Farthest Cluster Rank Summary Statistics by Hop Complexity ---
        count  mean  median  max  90th_percentile  <lambda_1>  <lambda_2>
n_hops                                                                   
2        1252  13.5     4.0  174             37.0        62.5       113.0
3         760  22.9    10.0  212             61.1        89.0       155.2
4         405  45.4    33.0  214             95.0       132.2       203.0


In [55]:
query_df['n_hops'].value_counts()

n_hops
2    15628
3     5147
4     1580
Name: count, dtype: int64

In [43]:
# ---- Phase C: oracle routing — top-CLUSTER_BUDGET PER gold cluster, then union ----
records = []
for qi, n_hops, gidx in items:
    gold_set, n_gold = set(gidx.tolist()), len(gidx)
    q = query_norm[qi]

    # --- 1. Standard Vector RAG Similarity ---
    full_retrieved_indices = list(full_topsets[qi])
    full_hits = len(full_topsets[qi] & gold_set)
    
    # Dot product calculation for Standard Vector RAG
    full_sims = doc_norm[full_retrieved_indices] @ q
    avg_full_cos_sim = float(np.mean(full_sims)) if len(full_sims) > 0 else 0.0

    # --- 2. Oracle Cluster Routing Similarity ---
    gold_codes = set(doc_label_codes[gidx].tolist())     # the fine clusters the router must guess
    retrieved = set()
    for c in gold_codes:
        c_doc_idx = cluster_to_doc_idx[c]
        c_sims = doc_norm[c_doc_idx] @ q
        k = min(CLUSTER_BUDGET, len(c_doc_idx))
        top_c = (c_doc_idx[np.argpartition(-c_sims, k-1)[:k]]
                 if k < len(c_doc_idx) else c_doc_idx)
        retrieved.update(top_c.tolist())

    oracle_retrieved_indices = list(retrieved)
    oracle_hits = len(retrieved & gold_set)
    
    # Dot product calculation for Oracle Cluster Routing
    oracle_sims = doc_norm[oracle_retrieved_indices] @ q
    avg_oracle_cos_sim = float(np.mean(oracle_sims)) if len(oracle_sims) > 0 else 0.0

    # --- 3. Append to Records ---
    records.append({
        "n_hops": n_hops,
        "n_gold": n_gold,
        "n_gold_clusters": len(gold_codes),
        "docs_retrieved": len(retrieved),
        "oracle_cluster_recall": oracle_hits / n_gold,
        "full_rag_recall":       full_hits  / n_gold,
        # New metrics:
        "oracle_avg_cosine":     avg_oracle_cos_sim,
        "full_rag_avg_cosine":   avg_full_cos_sim
    })

res_df = pd.DataFrame(records)
print(f"\nAvg docs pulled by oracle routing: {res_df['docs_retrieved'].mean():.1f} "
      f"(vs {FULL_BUDGET} for vector RAG) across "
      f"{res_df['n_gold_clusters'].mean():.1f} clusters/query on average.")


Avg docs pulled by oracle routing: 53.7 (vs 150 for vector RAG) across 2.1 clusters/query on average.


In [44]:
cols = ["oracle_cluster_recall", "full_rag_recall"]
sim_cols = ["oracle_avg_cosine", "full_rag_avg_cosine"]

# Base aggregation for Recall (multiplied by 100 for %)
agg = res_df.groupby("n_hops")[cols].mean().mul(100)

# Add the Cosine Similarities (kept as raw decimals 0.0 - 1.0)
agg = agg.join(res_df.groupby("n_hops")[sim_cols].mean())

# Add the count / metadata columns
agg = agg.join(res_df.groupby("n_hops")[["n_gold", "n_gold_clusters", "docs_retrieved"]].mean())

# Append the final "All" row calculation
agg.loc["All", cols] = res_df[cols].mean().mul(100).values
agg.loc["All", sim_cols] = res_df[sim_cols].mean().values
agg.loc["All", ["n_gold", "n_gold_clusters", "docs_retrieved"]] = \
    res_df[["n_gold", "n_gold_clusters", "docs_retrieved"]].mean().values

print("\n--- Summary Performance Evaluation Matrix ---")
print(agg.round(4))


--- Summary Performance Evaluation Matrix ---
        oracle_cluster_recall  full_rag_recall  oracle_avg_cosine  \
n_hops                                                              
2                     81.3498          79.4329             0.5179   
3                     78.0263          77.5439             0.5208   
4                     63.7654          52.1605             0.5026   
All                   77.3583          74.2691             0.5163   

        full_rag_avg_cosine  n_gold  n_gold_clusters  docs_retrieved  
n_hops                                                                
2                    0.5192  2.0000           1.7061         42.6518  
3                    0.5294  3.0000           2.3421         58.5526  
4                    0.5317  4.0000           3.1580         78.9506  
All                  0.5245  2.6496           2.1494         53.7340  


In [82]:
import os
import json
import numpy as np
from sklearn.preprocessing import normalize

SAVE_DIR = "../data/artifacts/clustering_output"
os.makedirs(SAVE_DIR, exist_ok=True)

# ---- 1. Queries in cosine space (rows align to query_emb / query_df order) ----
q_norm = normalize(query_emb, norm="l2").astype(np.float32)

# ---- 2. Coarse centroids (high-level, ~20) ----
coarse_centroids = results["coarse_model"].cluster_centers_
coarse_ids = [f"C{c}" for c in range(coarse_centroids.shape[0])]
coarse_norm = normalize(coarse_centroids, norm="l2").astype(np.float32)

# ---- 3. Fine centroids (low-level, ~200), flattened in C{coarse}_F{fine} order ----
fine_ids, fine_list = [], []
for c_id, f_model in results["fine_models"].items():
    for f_label, centroid in enumerate(f_model.cluster_centers_):
        fine_ids.append(f"C{c_id}_F{f_label}")
        fine_list.append(centroid)
fine_centroids = np.asarray(fine_list)
fine_norm = normalize(fine_centroids, norm="l2").astype(np.float32)

# ---- 4. Cosine similarities: rows = queries, cols = clusters ----
query_coarse_sims = q_norm @ coarse_norm.T     # (n_queries, n_coarse)
query_fine_sims   = q_norm @ fine_norm.T       # (n_queries, n_fine)

print(f"Coarse sims: {query_coarse_sims.shape}  (queries x {len(coarse_ids)} coarse)")
print(f"Fine sims  : {query_fine_sims.shape}  (queries x {len(fine_ids)} fine)")

# ---- 5. Optional: carry query identifiers for easy joining later ----
extra = {}
for col in ["question_id", "qid", "id"]:
    if col in query_df.columns:
        extra["query_ids"] = query_df[col].to_numpy()
        break

# ---- 6. Save (one self-describing .npz + a readable JSON of column order) ----
np.savez_compressed(
    os.path.join(SAVE_DIR, "query_cluster_sims.npz"),
    query_coarse_sims=query_coarse_sims,
    query_fine_sims=query_fine_sims,
    coarse_ids=np.array(coarse_ids),
    fine_ids=np.array(fine_ids),
    **extra,
)
with open(os.path.join(SAVE_DIR, "cluster_sim_columns.json"), "w") as f:
    json.dump({"coarse_ids": coarse_ids, "fine_ids": fine_ids}, f, indent=2)

print(f"Saved -> {SAVE_DIR}/query_cluster_sims.npz")

Coarse sims: (22355, 20)  (queries x 20 coarse)
Fine sims  : (22355, 228)  (queries x 228 fine)
Saved -> ../data/artifacts/clustering_output/query_cluster_sims.npz


In [83]:
import os, json
import numpy as np
from sklearn.preprocessing import normalize

SAVE_DIR = "../data/artifacts/clustering_output"
os.makedirs(SAVE_DIR, exist_ok=True)

# ---- Centroids in cosine space (identical column order to the query file) ----
coarse_centroids = results["coarse_model"].cluster_centers_
coarse_ids = [f"C{c}" for c in range(coarse_centroids.shape[0])]
coarse_norm = normalize(coarse_centroids, norm="l2").astype(np.float32)

fine_ids, fine_list = [], []
for c_id, f_model in results["fine_models"].items():
    for f_label, centroid in enumerate(f_model.cluster_centers_):
        fine_ids.append(f"C{c_id}_F{f_label}")
        fine_list.append(centroid)
fine_norm = normalize(np.asarray(fine_list), norm="l2").astype(np.float32)

# ---- Allocate outputs (rows = docs, aligned to doc_emb / docs_df order) ----
n_docs = len(doc_emb)
doc_coarse_sims = np.empty((n_docs, coarse_norm.shape[0]), dtype=np.float32)
doc_fine_sims   = np.empty((n_docs, fine_norm.shape[0]),   dtype=np.float32)

# ---- Chunked cosine so we never hold a full normalized copy of doc_emb ----
CH = 50_000
for s in range(0, n_docs, CH):
    e = min(s + CH, n_docs)
    block = normalize(doc_emb[s:e], norm="l2").astype(np.float32)
    doc_coarse_sims[s:e] = block @ coarse_norm.T
    doc_fine_sims[s:e]   = block @ fine_norm.T
    print(f"  processed {e:,}/{n_docs:,} docs", end="\r")

print(f"\nCoarse sims: {doc_coarse_sims.shape}   Fine sims: {doc_fine_sims.shape}")

# ---- Doc identifiers for joining (node_id), aligned to doc_emb order ----
node_ids = docs_df["node_id"].to_numpy() if "node_id" in docs_df.columns else np.arange(n_docs)

# ---- Save ----
np.savez_compressed(
    os.path.join(SAVE_DIR, "doc_cluster_sims.npz"),
    doc_coarse_sims=doc_coarse_sims,
    doc_fine_sims=doc_fine_sims,
    coarse_ids=np.array(coarse_ids),
    fine_ids=np.array(fine_ids),
    node_ids=node_ids,
)
mb = (doc_coarse_sims.nbytes + doc_fine_sims.nbytes) / 1e6
print(f"Saved -> {SAVE_DIR}/doc_cluster_sims.npz  (~{mb:.0f} MB uncompressed)")

  processed 101,962/101,962 docs
Coarse sims: (101962, 20)   Fine sims: (101962, 228)
Saved -> ../data/artifacts/clustering_output/doc_cluster_sims.npz  (~101 MB uncompressed)
